# Experiment 3 — Controlled Directional Perturbation via PCA Latent Shift

Assess sensitivity/uncertainty of DiT by measuring how much the **final image** changes when a mid-trajectory latent is shifted along an empirically-estimated **PCA direction of the latent space**, then denoised to completion.

This notebook follows the **shared cross-experiment configuration** in `project_setup.md` §2, so its numbers are directly comparable with Experiments 1 and 2. Settings fixed by that agreement are tagged **[shared]** in the config cell and must not be changed here alone.

**Protocol** (repeated independently for every class below)
1. Generate `M` independent base reverse trajectories per class from deterministic sha256-derived seeds; save latents at the shared checkpoint grid $s \in \{25, 75, 125, 150, 175, 225\}$ (fractions $0.1, 0.3, 0.5, 0.6, 0.7, 0.9$ of the 250 reverse steps) **[shared]**.
2. At each checkpoint, empirically estimate a local PCA basis of the latent space by generating an ensemble of `K_pca` independently-seeded full trajectories (same class) and collecting their latents at that checkpoint.
3. For each of the top PCA components, shift the base latent by $\alpha \in \{-3,-1,+1,+3\}$ standard deviations ($\sqrt{\text{eigenvalue}}$) along that component — a controlled, *directional* (not random) corruption, at the *same* timestep (no backward noise jump).
4. Continue denoising from that checkpoint **reusing the base trajectory's own per-step noise** **[shared]**, so the perturbation is the only thing that differs from the base run. A secondary, explicitly labelled `shared_fresh` ablation (every resume shares one independently drawn noise seed) is implemented but off by default.
5. Compare each perturbed final against the unperturbed **control final** ($\alpha=0$, same checkpoint and noise policy) via CLIP (OpenCLIP ViT-B/32, `openai`), DINOv2 (`vit_small_patch14_dinov2.lvd142m`), LPIPS (AlexNet) and latent MSE **[shared]**.
6. Report every perturbation magnitude twice: natively in PC-$\sigma$ units ($\alpha$), and normalised by the local per-checkpoint latent std **[shared]**, so this experiment's relative magnitudes can be plotted on the same axis as Exp 1/2's absolute-unit perturbations.

Under the default `reuse_base` policy the $\alpha=0$ control reproduces the base trajectory's own final image at every checkpoint (verified numerically during the run), which also removes the old caveat that controls differed from checkpoint to checkpoint.

**Class selection.** We run this protocol over the **KID difficulty tiers** — the 15 **hardest** (highest KID), 15 **medium**, and 15 **easiest** (lowest KID) classes — to test whether PCA-directional sensitivity tracks per-class generation difficulty. The 45 IDs come from the canonical shared tier list **[shared]** instead of being re-derived here; the config cell cross-checks them against `results/ensemble_K50_fid/kid_per_class.json` and reports any drift. Tier-aggregated plots cover all 45 classes; detailed image grids / diff maps are shown for one *spotlight* (median-KID) class per tier.


# 1. Setup


In [ ]:
%cd ../


In [ ]:
#!git clone https://github.com/facebookresearch/DiT.git
import DiT, os
os.chdir("DiT")

import hashlib
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torchvision.utils import make_grid, save_image

from diffusion import create_diffusion
from diffusers.models import AutoencoderKL
from download import find_model
from models import DiT_XL_2

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")
if device == "cpu":
    print("GPU not found. Using CPU instead.")


# 2. Load DiT-XL/2


In [ ]:
image_size = 256  #@param [256, 512]
# [shared] sd-vae-ft-mse, not -ema: the two fine-tunes have different reconstruction
# characteristics, so mixing them would offset latent statistics and every decoded-image
# metric relative to Exp 1/2 for reasons unrelated to the perturbation.
vae_model = "stabilityai/sd-vae-ft-mse"  #@param ["stabilityai/sd-vae-ft-mse", "stabilityai/sd-vae-ft-ema"]
latent_size = int(image_size) // 8

model = DiT_XL_2(input_size=latent_size).to(device)
state_dict = find_model(f"DiT-XL-2-{image_size}x{image_size}.pt")
model.load_state_dict(state_dict)
model.eval()
vae = AutoencoderKL.from_pretrained(vae_model).to(device)
print("Model + VAE ready.")


# 3. Config — KID difficulty tiers

Classes are taken from the same ranking used to build `results/ensemble_K50_fid/kid_difficulty_tiers.png`: 15 hardest / 15 medium / 15 easiest by per-class KID.


In [ ]:
# ============================================================================
# [shared] Cross-experiment configuration — see project_setup.md §2.
# Anything tagged [shared] is fixed by agreement across Exp 1/2/3; changing it
# here alone silently breaks comparability with the other two experiments.
# ============================================================================
SHARED_CONFIG_VERSION = "unified_v1"

# ---- Sampling [shared] ----
cfg_scale = 4.0  #@param {type:"slider", min:1, max:10, step:0.1}
num_sampling_steps = 250  #@param {type:"slider", min:10, max:1000, step:1}

# ---- Checkpoints / branch points [shared] ----
# Respaced reverse-step indices out of 250 = fracs {.1, .3, .5, .6, .7, .9}. Five of
# them are the grid Exp 1 and Exp 2 already agree on; s=150 (0.6) is the extra point
# that covers this experiment's mid-denoising range without dropping the extremes.
checkpoint_steps = (25, 75, 125, 150, 175, 225)  #@param {type:"raw"}

# ---- Trajectory structure [shared terminology] ----
# M = number of independent base trajectories per class (base-seed variance)
# K = number of perturbation copies per (base, checkpoint, perturbation type)
# The PCA shift is deterministic given the basis, so K = 1 by construction. M = 1 is
# stated explicitly rather than left implicit; raising it is a config change only,
# since every result is keyed by base index m.
M = 1  #@param {type:"integer"}
K = 1  # fixed by design: the directional shift has no per-copy randomness

# ---- Noise-reuse policy after perturbation [shared] ----
# "reuse_base"   : resume with the base trajectory's own per-step noise (primary).
#                  Everything but the intervention is held fixed, so the measured
#                  spread is perturbation effect only.
# "shared_fresh" : resume with one independently drawn shared noise seed. This
#                  conflates the perturbation with a different noise realisation,
#                  so it stays a labelled ablation and never the headline number.
NOISE_POLICY_PRIMARY = "reuse_base"
NOISE_POLICIES = ("reuse_base",)  #@param {type:"raw"}  # add "shared_fresh" to also run the ablation
shared_denoise_seed = 12345  #@param {type:"integer"}  # used by "shared_fresh" only

# ---- PCA basis (built per-checkpoint from an ensemble of extra seeded trajectories) ----
K_pca = 32  #@param {type:"integer"}              # #extra trajectories used to estimate PCA per checkpoint
pca_components = (0, 1, 2)  #@param {type:"raw"}  # which PCs to test (0 = top/dominant component)
alphas = (-3.0, -1.0, 1.0, 3.0)  #@param {type:"raw"}  # shift magnitude, in std (sqrt eigenvalue) units

# ---- Seeds [shared convention] ----
# Seeds are derived from a namespaced sha256 of their role rather than from a base
# offset, so any single trajectory — and therefore any single output image — can be
# regenerated in isolation months later.
SEED_NAMESPACE = f"exp3_pca/{SHARED_CONFIG_VERSION}"


def derive_seed(*parts, bits=31):
    """Deterministic seed for a named role (stable across machines and runs)."""
    key = "|".join(str(p) for p in (SEED_NAMESPACE, *parts))
    return int(hashlib.sha256(key.encode()).hexdigest(), 16) % (2 ** bits)


# ---- Class set [shared] ----
# The 45 tier IDs have one canonical definition for the whole project and must be
# referenced verbatim, never re-digitised from the KID chart. Point
# CANONICAL_TIERS_PATH at that file (YAML: tier name -> list of class ids) or paste the
# lists straight into CANONICAL_TIERS. Without either we re-derive from
# kid_per_class.json and say so loudly.
CANONICAL_TIERS_PATH = Path("../configs/kid_difficulty_tiers.yaml")
CANONICAL_TIERS = None  #@param {type:"raw"}

# ---- KID difficulty tiers (same split as kid_difficulty_tiers.png) ----
KID_JSON = Path("../results/ensemble_K50_fid/kid_per_class.json")
IMAGENET_IDX = Path("../results/ensemble_K50_fid/imagenet_class_index.json")

with open(KID_JSON) as f:
    kid_per_class = {int(k): v for k, v in json.load(f).items()}
with open(IMAGENET_IDX) as f:
    _idx = json.load(f)
class_names = {i: _idx[str(i)][1] for i in range(len(_idx))}

kid_ranked = sorted(kid_per_class.keys(), key=lambda c: kid_per_class[c]["kid"], reverse=True)
n_ranked = len(kid_ranked)
TIER_SIZE = min(15, n_ranked // 3 if n_ranked >= 3 else n_ranked)

mid_start = max(0, n_ranked // 2 - TIER_SIZE // 2)
derived_tiers = {
    "hardest": kid_ranked[:TIER_SIZE],
    "medium": kid_ranked[mid_start: mid_start + TIER_SIZE],
    "easiest": kid_ranked[-TIER_SIZE:] if TIER_SIZE > 0 else [],
}


def load_canonical_tiers():
    """The shared 45-class tier lists, or (None, None) if they are not available here."""
    if CANONICAL_TIERS is not None:
        return {k: [int(c) for c in v] for k, v in CANONICAL_TIERS.items()}, "CANONICAL_TIERS (inline)"
    if CANONICAL_TIERS_PATH.exists():
        import yaml

        with open(CANONICAL_TIERS_PATH) as f:
            raw = yaml.safe_load(f)
        raw = raw.get("tiers", raw)
        return {k: [int(c) for c in raw[k]] for k in derived_tiers}, str(CANONICAL_TIERS_PATH)
    return None, None


tiers, TIER_SOURCE = load_canonical_tiers()
if tiers is None:
    tiers, TIER_SOURCE = derived_tiers, "re-derived from kid_per_class.json (NOT canonical)"
    print(
        f"WARNING: no canonical tier list at {CANONICAL_TIERS_PATH}. Falling back to the\n"
        "         KID re-derivation. Copy the 45 IDs from the shared project config into\n"
        "         that file before quoting these numbers against Exp 1/2.\n"
    )
else:
    for name in derived_tiers:
        drift = sorted(set(tiers[name]) ^ set(derived_tiers[name]))
        if drift:
            print(f"NOTE: canonical '{name}' tier differs from the local KID re-derivation at {drift}")
    print(f"Class set: canonical list from {TIER_SOURCE}\n")

hardest_classes = tiers["hardest"]
medium_classes = tiers["medium"]
easiest_classes = tiers["easiest"]
tier_colors = {
    "hardest": "crimson",
    "medium": "goldenrod",
    "easiest": "seagreen",
}

# One spotlight class per tier (median-KID within the tier) for detailed visuals
spotlight_classes = {name: cls_list[len(cls_list) // 2] for name, cls_list in tiers.items()}

# Checkpoints shown in spotlight image grids / diff maps (must be on the shared grid)
spotlight_show_steps = (125, 150)  #@param {type:"raw"}

class_labels = hardest_classes + medium_classes + easiest_classes
class_to_tier = {}
for name, cls_list in tiers.items():
    for c in cls_list:
        class_to_tier[c] = name

# Runs under the shared config live in their own subdirectory: the checkpoint grid, the
# VAE and the noise policy all changed, so pre-unification caches must not be picked up.
RESULTS_DIR = Path("../results/exp3_pca_perturbation") / SHARED_CONFIG_VERSION
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

diffusion = create_diffusion(str(num_sampling_steps))
shape = (1, 4, latent_size, latent_size)
T = diffusion.num_timesteps
LATENT_DIM = 4 * latent_size * latent_size
assert T == num_sampling_steps, (T, num_sampling_steps)

checkpoint_steps = tuple(int(s) for s in checkpoint_steps)
assert all(0 < s < T for s in checkpoint_steps), "checkpoints must allow at least one more denoising step"
assert set(NOISE_POLICIES) <= {"reuse_base", "shared_fresh"}, NOISE_POLICIES
assert NOISE_POLICY_PRIMARY in NOISE_POLICIES, "the primary noise policy must actually be run"

spotlight_show_steps = tuple(int(s) for s in spotlight_show_steps)
assert all(s in checkpoint_steps for s in spotlight_show_steps), (
    f"spotlight_show_steps={spotlight_show_steps} must be subset of checkpoint_steps={checkpoint_steps}"
)

print(f"shared config {SHARED_CONFIG_VERSION} -> {RESULTS_DIR}")
print(f"steps={T}, cfg={cfg_scale}, vae={vae_model}")
print(f"checkpoints={checkpoint_steps} (fracs {tuple(round(s / T, 2) for s in checkpoint_steps)})")
print(f"M={M} base trajectories/class, K={K} perturbation copies/condition")
print(f"noise policies={NOISE_POLICIES} (primary={NOISE_POLICY_PRIMARY}), shared_denoise_seed={shared_denoise_seed}")
print(f"K_pca={K_pca}, pca_components={pca_components}, alphas={alphas}")
print(f"\nclass set: {TIER_SOURCE} | TIER_SIZE={TIER_SIZE} | total classes to run = {len(class_labels)}")
for name, cls_list in tiers.items():
    print(f"\n=== {name} ({len(cls_list)}) | spotlight={spotlight_classes[name]} ({class_names.get(spotlight_classes[name], '?')}) ===")
    for c in cls_list:
        print(f"  class {c:>4}  KID={kid_per_class[c]['kid']:.4f}  {class_names.get(c, '?')}")


# 4. Diffusion helpers

Indexing convention:
- Reverse step index $s=0$: initial noise (input to the first reverse step).
- After $s$ reverse steps, latent `traj[s]` is the input for spaced timestep index $i = T-1-s$.
- The PCA perturbation shifts the latent *at the same reverse-step* $s$ (no time jump); only `denoise_from` is needed to resume.

Trajectories draw their reverse-step noise from an explicit, seed-derived sequence rather than from the global RNG inside `p_sample`. That is what makes the **[shared]** `reuse_base` policy possible: resuming from `traj[s]` with the same trajectory's noise is a bit-for-bit continuation of that run, so an unperturbed resume lands back on `traj[T]`.


In [ ]:
def spaced_index_at_step(s):
    """Spaced diffusion index used as model-t when the latent is traj[s] (s < T)."""
    return T - 1 - s


def decode_latents(latents):
    return vae.decode(latents / 0.18215).sample


def save_uint8_image(tensor_chw, path):
    x = torch.clamp(127.5 * tensor_chw + 128.0, 0, 255)
    x = x.permute(1, 2, 0).to("cpu", dtype=torch.uint8).numpy()
    Image.fromarray(x).save(path)


def p_sample_with_noise(diffusion, model_fn, x, t, step_noise, model_kwargs):
    """Like diffusion.p_sample, but uses provided step_noise instead of randn."""
    out = diffusion.p_mean_variance(
        model_fn, x, t, clip_denoised=False, model_kwargs=model_kwargs
    )
    nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
    sample = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * step_noise
    return {"sample": sample, "pred_xstart": out["pred_xstart"]}


def precompute_step_noise(noise_seed, start_step=0):
    """Per reverse-step eps for CFG batch (2, C, H, W), indexed by reverse step_i."""
    g = torch.Generator(device=device)
    g.manual_seed(int(noise_seed))
    noises = []
    for _ in range(T):
        noises.append(torch.randn(2, 4, latent_size, latent_size, generator=g, device=device))
    return noises[start_step:]


def cfg_kwargs(class_label):
    y = torch.tensor([class_label, 1000], device=device)
    return dict(y=y, cfg_scale=cfg_scale)


def base_traj_seeds(class_label, m):
    """(init_seed, noise_seed) for base trajectory m of a class."""
    return derive_seed("base_init", class_label, m), derive_seed("base_noise", class_label, m)


def pca_traj_seeds(class_label, k):
    """(init_seed, noise_seed) for PCA-fitting trajectory k of a class."""
    return derive_seed("pca_init", class_label, k), derive_seed("pca_noise", class_label, k)


def generate_trajectory(class_label, init_seed, noise_seed, save_steps):
    """Run one full reverse path; return dict step -> latent (1,C,H,W) on CPU.

    traj[0] = initial noise; traj[T] = final latent. The reverse-step noise comes from
    `noise_seed` rather than the global RNG, so the same sequence can be replayed when
    resuming from any checkpoint.
    """
    save_set = set(save_steps) | {0, T}
    g = torch.Generator(device=device)
    g.manual_seed(int(init_seed))
    z = torch.randn(*shape, generator=g, device=device)
    img = torch.cat([z, z], dim=0)
    model_kwargs = cfg_kwargs(class_label)
    step_noises = precompute_step_noise(noise_seed, start_step=0)

    traj = {}
    if 0 in save_set:
        traj[0] = img[:1].detach().cpu()

    indices = list(range(T))[::-1]
    for step_i, i in enumerate(tqdm(indices, desc="trajectory", leave=False)):
        t = torch.tensor([i, i], device=device)
        out = p_sample_with_noise(
            diffusion,
            model.forward_with_cfg,
            img,
            t,
            step_noises[step_i],
            model_kwargs,
        )
        img = out["sample"]
        s = step_i + 1
        if s in save_set:
            traj[s] = img[:1].detach().cpu()

    return traj


def resume_step_noises(policy, base_noise_seed, start_step):
    """[shared] Reverse-step noise for a resume, per the agreed noise-reuse policy."""
    if policy == "reuse_base":
        return precompute_step_noise(base_noise_seed, start_step=start_step)
    if policy == "shared_fresh":
        return precompute_step_noise(shared_denoise_seed, start_step=start_step)
    raise ValueError(f"unknown noise policy {policy!r}")


def denoise_from(x_start, start_step, class_label, step_noises):
    """Continue reverse diffusion from reverse-step start_step to the end.

    step_noises[k] is used at reverse step start_step + k (CFG-shaped).
    Returns final latent (1,C,H,W) on CPU.
    """
    img = x_start.to(device)
    img = torch.cat([img, img], dim=0)
    model_kwargs = cfg_kwargs(class_label)
    indices = list(range(T))[::-1]

    for k, step_i in enumerate(range(start_step, T)):
        i = indices[step_i]
        t = torch.tensor([i, i], device=device)
        out = p_sample_with_noise(
            diffusion,
            model.forward_with_cfg,
            img,
            t,
            step_noises[k],
            model_kwargs,
        )
        img = out["sample"]

    return img[:1].detach().cpu()


print("Diffusion helpers ready.")


# 5. Feature / distance metric helpers (CLIP, DINOv2, LPIPS, latent MSE)

Unlike Experiment 1 (ensemble spread across seeds), each PCA-perturbation condition here is a single deterministic trajectory. We therefore report **pairwise distance** between each perturbed final and the unperturbed **control final** at the same checkpoint and noise policy, rather than ensemble variance.

The encoders and their preprocessing are **[shared]** across Exp 1/2/3: OpenCLIP `ViT-B-32` / `openai` at 224 bicubic, DINO**v2** `vit_small_patch14_dinov2.lvd142m` via `timm` (DINOv1 and DINOv2 embeddings are not interchangeable), and LPIPS with the AlexNet backbone. Package versions are captured into `ENV_VERSIONS` and written into every results JSON, since unpinned encoders are what caused the three experiments to diverge in the first place.


In [ ]:
import importlib.metadata as importlib_metadata

import lpips
import open_clip
import timm


# ---- CLIP [shared]: open_clip ViT-B-32 / openai, images resized 224 bicubic ----
CLIP_MODEL_NAME, CLIP_PRETRAINED = "ViT-B-32", "openai"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
)
clip_model = clip_model.to(device).eval()

# The shared spec pins preprocessing as well as weights: the same openai checkpoint
# loaded through a different library can resize or normalise differently.
_clip_resize = next(
    (t for t in clip_preprocess.transforms if type(t).__name__ == "Resize"), None
)
assert _clip_resize is not None, clip_preprocess
assert _clip_resize.size in (224, [224, 224], (224, 224)), _clip_resize.size
assert "bicubic" in str(_clip_resize.interpolation).lower(), _clip_resize.interpolation

# ---- DINOv2 [shared]: timm vit_small_patch14_dinov2.lvd142m ----
# DINOv1 and DINOv2 features are not comparable, so both the checkpoint and the
# preprocessing below mirror src/uncertainty/features.TimmEncoder exactly.
DINO_MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
dino_model = timm.create_model(DINO_MODEL_NAME, pretrained=True, num_classes=0)
dino_model = dino_model.eval().to(device)
_dino_cfg = timm.data.resolve_data_config({}, model=dino_model)
DINO_INPUT = _dino_cfg["input_size"][-1]
_dino_mean = torch.tensor(_dino_cfg["mean"], device=device).view(1, 3, 1, 1)
_dino_std = torch.tensor(_dino_cfg["std"], device=device).view(1, 3, 1, 1)

# ---- LPIPS [shared] ----
lpips_fn = lpips.LPIPS(net="alex").to(device).eval()


def _pkg_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"


# Written into every results JSON so a number can always be traced back to the exact
# encoder stack that produced it.
ENV_VERSIONS = {
    "torch": torch.__version__,
    "timm": _pkg_version("timm"),
    "open_clip_torch": _pkg_version("open_clip_torch"),
    "lpips": _pkg_version("lpips"),
    "diffusers": _pkg_version("diffusers"),
    "clip": f"{CLIP_MODEL_NAME}/{CLIP_PRETRAINED} @224 bicubic",
    "dino": f"timm:{DINO_MODEL_NAME} @{DINO_INPUT} bicubic",
    "lpips_net": "alex",
    "vae": vae_model,
}


def _images_to_pil_list(imgs_bchw):
    """imgs in [-1,1] -> list of PIL RGB."""
    x = torch.clamp(imgs_bchw * 0.5 + 0.5, 0, 1)
    out = []
    for i in range(x.shape[0]):
        arr = (x[i].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        out.append(Image.fromarray(arr))
    return out


@torch.no_grad()
def embed_clip(imgs_bchw):
    pils = _images_to_pil_list(imgs_bchw)
    tensors = torch.stack([clip_preprocess(p) for p in pils]).to(device)
    feats = clip_model.encode_image(tensors)
    return F.normalize(feats.float(), dim=-1).cpu()


@torch.no_grad()
def embed_dino(imgs_bchw):
    x = (imgs_bchw.to(device).float().clamp(-1, 1) + 1) / 2
    x = F.interpolate(x, size=(DINO_INPUT, DINO_INPUT), mode="bicubic", align_corners=False)
    feats = dino_model((x - _dino_mean) / _dino_std)
    return F.normalize(feats.float(), dim=-1).cpu()


@torch.no_grad()
def pairwise_metrics(latent_a, latent_b):
    """Compare two single final latents (1,C,H,W) CPU (e.g. perturbed vs control).

    Returns (metrics_dict, img_a, img_b) with images decoded to (C,H,W) in [-1,1] on CPU.
    """
    imgs = decode_latents(torch.cat([latent_a, latent_b], dim=0).to(device)).cpu()
    img_a, img_b = imgs[0:1], imgs[1:2]

    clip_f = embed_clip(imgs)
    dino_f = embed_dino(imgs)
    clip_cos_dist = float(1.0 - (clip_f[0] @ clip_f[1]))
    dino_cos_dist = float(1.0 - (dino_f[0] @ dino_f[1]))
    lpips_d = float(lpips_fn(img_a.to(device), img_b.to(device)).item())
    latent_mse = float(F.mse_loss(latent_a.reshape(-1), latent_b.reshape(-1)).item())
    pixel_mse = float(F.mse_loss(img_a, img_b).item())

    metrics = {
        "clip_cos_dist": clip_cos_dist,
        "dino_cos_dist": dino_cos_dist,
        "lpips": lpips_d,
        "latent_mse": latent_mse,
        "pixel_mse": pixel_mse,
    }
    return metrics, img_a[0], img_b[0]


print("Metric helpers ready (CLIP / DINOv2 / LPIPS, pairwise-vs-control).")
for _k, _v in ENV_VERSIONS.items():
    print(f"  {_k:<16} {_v}")


# 6. Per-class experiment pipeline

Helpers that run the full protocol for one ImageNet class and write results under `results/exp3_pca_perturbation/unified_v1/class_XXXX/`. Existing caches (`traj_m*.pt`, `pca_ensemble_*.pt`, `metrics_*.json`) are reused so re-runs skip finished work.

Each result is keyed by `{noise_policy}/m{base}/s{checkpoint}_pc{component}_a{alpha}` and carries its **[shared]** magnitude report (`alpha`, `pc_sigma`, `pert_rms`, `latent_std`, `alpha_std_units`) alongside the distances. Under `reuse_base` the pipeline also records `control_drift_vs_base_final`: the max absolute difference between the unperturbed resume and the base trajectory's own final latent, which should be ~0 and is the check that the noise replay is intact.

Pre-unification runs live in the parent directory and are not read by this notebook — the checkpoint grid, VAE and noise policy all differ, so those caches are not reusable.


In [ ]:
def fit_pca(latents_kchw, n_components):
    """Fit PCA on an ensemble of latents via SVD of the centered data matrix."""
    K = latents_kchw.shape[0]
    flat = latents_kchw.reshape(K, -1).double()
    mean = flat.mean(dim=0)
    centered = flat - mean
    _, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    eigvals = (S ** 2) / (K - 1)
    total_var = eigvals.sum()

    n = min(n_components, Vh.shape[0])
    components = Vh[:n]
    std = torch.sqrt(eigvals[:n].clamp(min=0))
    explained_ratio = (eigvals[:n] / total_var).float()

    return {
        "mean": mean.float().reshape(latents_kchw.shape[1:]),
        "components": components.float().reshape(n, *latents_kchw.shape[1:]),
        "std": std.float(),
        "explained_variance_ratio": explained_ratio,
    }


def pca_perturb(x_s, basis, component_idx, alpha):
    """Shift latent x_s (1,C,H,W CPU) by alpha std-units along PCA component."""
    v = basis["components"][component_idx]
    std = basis["std"][component_idx].item()
    shift = (alpha * std) * v
    return (x_s.squeeze(0) + shift).unsqueeze(0)


def _log(msg):
    print(msg, flush=True)


def magnitude_report(alpha, pc_sigma, latent_std):
    """[shared] Native magnitude plus the same shift expressed in local-latent-std units.

    Exp 1/2 parametrise perturbations in absolute latent units and this one in PC-sigma
    units, so neither can be read off the other. Dividing the per-element RMS of the
    actual shift by the latent std at that checkpoint puts them on one axis: an absolute
    std=0.1 gaussian is 0.1 / latent_std in the same units.
    """
    l2 = abs(float(alpha)) * float(pc_sigma)
    rms = l2 / math.sqrt(LATENT_DIM)
    return {
        "alpha": float(alpha),
        "pc_sigma": float(pc_sigma),
        "pert_l2": l2,
        "pert_rms": rms,
        "latent_std": float(latent_std),
        "alpha_std_units": rms / float(latent_std),
    }


def metric_key(s, c, alpha, m=0, policy=None):
    """Result key: noise policy / base trajectory / checkpoint / PC / alpha."""
    return f"{policy or NOISE_POLICY_PRIMARY}/m{m}/s{s}_pc{c + 1}_a{alpha:+.1f}"


def _save_spotlight_pngs(class_label, samples_dir, show_steps, pca_basis, control_finals, exp3_results):
    """Write grid + diffmap PNGs for the requested checkpoint steps only."""
    for s in show_steps:
        with torch.no_grad():
            control_img = decode_latents(control_finals[s].to(device))[0].cpu()
        for c in pca_components:
            imgs = [control_img] + [exp3_results[(s, c, a)]["img"] for a in alphas]
            panel = torch.stack(imgs, dim=0)
            grid = make_grid(panel, nrow=len(panel), normalize=True, value_range=(-1, 1))
            plt.figure(figsize=(3 * len(panel), 3.2))
            plt.imshow(grid.permute(1, 2, 0).numpy())
            plt.axis("off")
            plt.title(f"class {class_label} | s={s}, PC{c + 1}")
            plt.savefig(samples_dir / f"grid_s{s}_pc{c + 1}.png", dpi=140, bbox_inches="tight")
            plt.close()

            fig, axs = plt.subplots(1, len(alphas) + 1, figsize=(3.2 * (len(alphas) + 1), 3.4))
            axs[0].imshow((control_img.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
            axs[0].set_title("control")
            axs[0].axis("off")
            im = None
            for ax, a in zip(axs[1:], alphas):
                pert_img = exp3_results[(s, c, a)]["img"]
                diff = (pert_img - control_img).abs().mean(dim=0).numpy()
                im = ax.imshow(diff, cmap="magma")
                ax.set_title(f"|d| a={a:+.1f}")
                ax.axis("off")
            fig.colorbar(im, ax=axs[-1], fraction=0.046, pad=0.04)
            fig.suptitle(f"|pert-ctrl| | class {class_label} s={s} PC{c + 1}")
            plt.tight_layout()
            fig.savefig(samples_dir / f"diffmap_s{s}_pc{c + 1}.png", dpi=140, bbox_inches="tight")
            plt.close(fig)


def save_spotlight_visuals(class_label, show_steps=None):
    """Generate grid/diffmap PNGs at selected checkpoints ONLY.

    Uses cached traj + PCA ensemble (no full-class re-run). Metrics JSON is untouched.
    Typical cost: len(show_steps) checkpoints x (3 PCs x 4 alphas) short denoises.
    """
    if show_steps is None:
        show_steps = spotlight_show_steps
    show_steps = tuple(int(s) for s in show_steps)

    samples_dir = RESULTS_DIR / f"class_{class_label:04d}"
    metrics_path = samples_dir / "metrics_pca_perturbation.json"
    if not metrics_path.exists():
        raise FileNotFoundError(
            f"No metrics for class {class_label}. Run §7 first for this class."
        )

    grids_ok = all(
        (samples_dir / f"grid_s{s}_pc{c + 1}.png").exists()
        for s in show_steps
        for c in pca_components
    )
    if grids_ok:
        _log(f"  [{class_label}] spotlight PNGs already exist for s={show_steps}")
        return

    traj_path = samples_dir / "traj_m0.pt"
    pca_ensemble_path = samples_dir / f"pca_ensemble_K{K_pca}.pt"
    if not traj_path.exists() or not pca_ensemble_path.exists():
        raise FileNotFoundError(
            f"Missing traj or PCA ensemble cache for class {class_label}."
        )

    _log(f"  [{class_label}] rendering spotlight visuals at s={show_steps} (metrics already saved)")

    payload = torch.load(traj_path, map_location="cpu", weights_only=False)
    traj = {int(k): v for k, v in payload["traj"].items()}
    payload = torch.load(pca_ensemble_path, map_location="cpu", weights_only=False)
    pca_latents = {int(k): v for k, v in payload["pca_latents"].items()}

    n_components_needed = max(pca_components) + 1
    pca_basis = {s: fit_pca(pca_latents[s], n_components_needed) for s in checkpoint_steps}

    _, base_noise_seed = base_traj_seeds(class_label, 0)
    control_finals = {}
    resume_noises_by_step = {}
    for s in show_steps:
        noises = resume_step_noises(NOISE_POLICY_PRIMARY, base_noise_seed, s)
        resume_noises_by_step[s] = noises
        control_finals[s] = denoise_from(traj[s], s, class_label, noises)

    exp3_results = {}
    n_jobs = len(show_steps) * len(pca_components) * len(alphas)
    pbar = tqdm(total=n_jobs, desc=f"vis class {class_label}", leave=True)
    for s in show_steps:
        x_s = traj[s]
        basis = pca_basis[s]
        resume_noises = resume_noises_by_step[s]
        control_final = control_finals[s]
        for c in pca_components:
            for alpha in alphas:
                x_pert = pca_perturb(x_s, basis, c, alpha)
                final = denoise_from(x_pert, s, class_label, resume_noises)
                metrics, img_pert, _ = pairwise_metrics(final, control_final)
                exp3_results[(s, c, alpha)] = {"metrics": metrics, "img": img_pert}
                pbar.update(1)
    pbar.close()

    _save_spotlight_pngs(class_label, samples_dir, show_steps, pca_basis, control_finals, exp3_results)
    _log(f"  [{class_label}] saved spotlight PNGs for s={show_steps}")


def run_class_experiment(class_label):
    """Full Exp3 protocol for one class. Returns metrics dict.

    If metrics_pca_perturbation.json exists, returns it immediately (no GPU re-run).
    """
    samples_dir = RESULTS_DIR / f"class_{class_label:04d}"
    samples_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = samples_dir / "metrics_pca_perturbation.json"

    if metrics_path.exists():
        with open(metrics_path) as f:
            cached = json.load(f)
        _log(f"  [{class_label}] SKIP (cached metrics)")
        return cached

    # ---- base trajectories (M per class, each resumable with its own noise) ----
    base_trajs, base_noise_seeds = {}, {}
    for m in range(M):
        init_seed, noise_seed = base_traj_seeds(class_label, m)
        base_noise_seeds[m] = noise_seed
        traj_path = samples_dir / f"traj_m{m}.pt"
        if traj_path.exists():
            payload = torch.load(traj_path, map_location="cpu", weights_only=False)
            base_trajs[m] = {int(k): v for k, v in payload["traj"].items()}
            _log(f"  [{class_label}] loaded base traj m={m} cache")
        else:
            _log(f"  [{class_label}] generating base trajectory m={m} ({T} steps)...")
            base_trajs[m] = generate_trajectory(
                class_label, init_seed, noise_seed, save_steps=list(checkpoint_steps)
            )
            torch.save(
                {
                    "traj": base_trajs[m],
                    "class_label": class_label,
                    "base_index": m,
                    "init_seed": init_seed,
                    "noise_seed": noise_seed,
                    "num_sampling_steps": T,
                    "cfg_scale": cfg_scale,
                    "shared_config_version": SHARED_CONFIG_VERSION,
                },
                traj_path,
            )
            _log(f"  [{class_label}] saved base traj m={m}")

    with torch.no_grad():
        ref_img = decode_latents(base_trajs[0][T].to(device))[0].cpu()
    save_uint8_image(ref_img, samples_dir / "reference_final.png")

    # ---- PCA ensemble ----
    pca_ensemble_path = samples_dir / f"pca_ensemble_K{K_pca}.pt"
    if pca_ensemble_path.exists():
        payload = torch.load(pca_ensemble_path, map_location="cpu", weights_only=False)
        pca_latents = {int(k): v for k, v in payload["pca_latents"].items()}
        _log(f"  [{class_label}] loaded PCA ensemble cache (K={K_pca})")
    else:
        _log(f"  [{class_label}] building PCA ensemble: {K_pca} trajectories (slow)...")
        pca_seeds = [pca_traj_seeds(class_label, k) for k in range(K_pca)]
        per_seed_traj = []
        for init_seed, noise_seed in tqdm(pca_seeds, desc=f"PCA ens class {class_label}", leave=True):
            per_seed_traj.append(
                generate_trajectory(
                    class_label, init_seed, noise_seed, save_steps=list(checkpoint_steps)
                )
            )
        pca_latents = {
            s: torch.cat([t[s] for t in per_seed_traj], dim=0) for s in checkpoint_steps
        }
        torch.save(
            {
                "pca_latents": pca_latents,
                "K_pca": K_pca,
                "pca_seeds": pca_seeds,
                "seed_namespace": SEED_NAMESPACE,
                "class_label": class_label,
                "checkpoint_steps": list(checkpoint_steps),
            },
            pca_ensemble_path,
        )
        _log(f"  [{class_label}] saved PCA ensemble")

    n_components_needed = max(pca_components) + 1
    pca_basis = {s: fit_pca(pca_latents[s], n_components_needed) for s in checkpoint_steps}
    torch.save(pca_basis, samples_dir / f"pca_basis_K{K_pca}.pt")
    # [shared] local scale that every perturbation magnitude is reported against
    latent_std = {s: float(pca_latents[s].std()) for s in checkpoint_steps}
    _log(f"  [{class_label}] fitted PCA basis")

    n_jobs = len(NOISE_POLICIES) * M * len(checkpoint_steps) * len(pca_components) * len(alphas)
    _log(f"  [{class_label}] PCA perturbation sweep: {n_jobs} jobs...")
    exp3_results = {}
    control_drift = {}
    pbar = tqdm(total=n_jobs, desc=f"pert class {class_label}", leave=True)
    for policy in NOISE_POLICIES:
        for m in range(M):
            traj = base_trajs[m]
            for s in checkpoint_steps:
                resume_noises = resume_step_noises(policy, base_noise_seeds[m], s)
                control_final = denoise_from(traj[s], s, class_label, resume_noises)
                if policy == "reuse_base":
                    # An unperturbed reuse_base resume has to land back on this
                    # trajectory's own final latent. Anything other than ~0 means the
                    # noise replay is broken and every distance below is meaningless.
                    control_drift[f"m{m}/s{s}"] = float((control_final - traj[T]).abs().max())
                for c in pca_components:
                    for alpha in alphas:
                        x_pert = pca_perturb(traj[s], pca_basis[s], c, alpha)
                        final = denoise_from(x_pert, s, class_label, resume_noises)
                        metrics, _, _ = pairwise_metrics(final, control_final)
                        metrics.update(
                            magnitude_report(alpha, pca_basis[s]["std"][c].item(), latent_std[s])
                        )
                        exp3_results[metric_key(s, c, alpha, m=m, policy=policy)] = metrics
                        pbar.update(1)
    pbar.close()

    if control_drift:
        _log(f"  [{class_label}] reuse_base control drift vs base final: max |d|={max(control_drift.values()):.2e}")

    metrics_out = {
        "config": {
            "shared_config_version": SHARED_CONFIG_VERSION,
            "class_label": class_label,
            "tier": class_to_tier.get(class_label),
            "kid": kid_per_class.get(class_label, {}).get("kid"),
            "checkpoint_steps": list(checkpoint_steps),
            "M": M,
            "K": K,
            "noise_policies": list(NOISE_POLICIES),
            "noise_policy_primary": NOISE_POLICY_PRIMARY,
            "shared_denoise_seed": shared_denoise_seed,
            "seed_namespace": SEED_NAMESPACE,
            "base_seeds": {str(m): list(base_traj_seeds(class_label, m)) for m in range(M)},
            "pca_components": list(pca_components),
            "alphas": list(alphas),
            "K_pca": K_pca,
            "latent_std_by_step": {str(s): latent_std[s] for s in checkpoint_steps},
            "cfg_scale": cfg_scale,
            "num_sampling_steps": T,
            "vae": vae_model,
            "env": ENV_VERSIONS,
        },
        "control_drift_vs_base_final": control_drift,
        "exp3": exp3_results,
    }
    with open(metrics_path, "w") as f:
        json.dump(metrics_out, f, indent=2)
    _log(f"  [{class_label}] wrote {metrics_path}")
    return metrics_out


print("run_class_experiment() + save_spotlight_visuals() ready.")


# 7. Run all KID-tier classes

Loops over hardest / medium / easiest (45 classes total). **Finished classes are skipped instantly** via cached `metrics_pca_perturbation.json` — re-running this cell only processes classes still missing metrics.

Cost per class scales as `K_pca` full trajectories (the PCA basis) plus `len(NOISE_POLICIES) x M x len(checkpoint_steps) x (1 + len(pca_components) x len(alphas))` partial denoises. The shared six-point checkpoint grid makes this ~50% more expensive per class than the old four-point one.

Spotlight PNGs (§9) are generated separately and do not trigger a full re-run.


In [ ]:
all_metrics = {}  # class_label -> metrics_out

# Load any class that already finished (§7 is safe to re-run: cached classes are instant)
done = [
    c for c in class_labels
    if (RESULTS_DIR / f"class_{c:04d}" / "metrics_pca_perturbation.json").exists()
]
pending = [c for c in class_labels if c not in done]
print(
    f"Progress: {len(done)}/{len(class_labels)} classes done, {len(pending)} remaining",
    flush=True,
)
if pending:
    print("Next pending:", pending[:10], ("..." if len(pending) > 10 else ""), flush=True)

if device == "cuda":
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU mem free={free/1e9:.2f}G / total={total/1e9:.2f}G", flush=True)

for tier_name, cls_list in tiers.items():
    for class_label in tqdm(cls_list, desc=f"tier={tier_name}"):
        print(
            f"\n>>> class {class_label} ({class_names.get(class_label, '?')}) "
            f"| tier={tier_name} | KID={kid_per_class[class_label]['kid']:.4f}",
            flush=True,
        )
        try:
            all_metrics[class_label] = run_class_experiment(class_label)
        except KeyboardInterrupt:
            print(
                f"\nInterrupted during class {class_label}. Re-run THIS cell only.",
                flush=True,
            )
            raise

# Aggregate summary JSON (loads from disk for classes skipped above)
for c in class_labels:
    if c not in all_metrics:
        with open(RESULTS_DIR / f"class_{c:04d}" / "metrics_pca_perturbation.json") as f:
            all_metrics[c] = json.load(f)

summary = {
    "tiers": {k: list(v) for k, v in tiers.items()},
    "spotlight_classes": spotlight_classes,
    "spotlight_show_steps": list(spotlight_show_steps),
    "config": {
        "shared_config_version": SHARED_CONFIG_VERSION,
        "checkpoint_steps": list(checkpoint_steps),
        "M": M,
        "K": K,
        "noise_policies": list(NOISE_POLICIES),
        "noise_policy_primary": NOISE_POLICY_PRIMARY,
        "shared_denoise_seed": shared_denoise_seed,
        "seed_namespace": SEED_NAMESPACE,
        "pca_components": list(pca_components),
        "alphas": list(alphas),
        "K_pca": K_pca,
        "cfg_scale": cfg_scale,
        "num_sampling_steps": T,
        "vae": vae_model,
        "class_set_source": TIER_SOURCE,
        "env": ENV_VERSIONS,
    },
    "per_class": {
        str(c): {
            "tier": class_to_tier[c],
            "kid": kid_per_class[c]["kid"],
            "name": class_names.get(c, "?"),
            "exp3": all_metrics[c]["exp3"],
        }
        for c in class_labels
        if c in all_metrics
    },
}
summary_path = RESULTS_DIR / "summary_all_tiers.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nWrote {summary_path} ({len(all_metrics)} classes)", flush=True)


# 8. Tier-aggregated plots

Mean ± std of sensitivity metrics across classes **within each KID difficulty tier**, vs. checkpoint and vs. PCA component / magnitude. Colors match `kid_difficulty_tiers.png` (crimson / goldenrod / seagreen).


In [ ]:
metric_keys = ["clip_cos_dist", "dino_cos_dist", "lpips", "latent_mse"]

# Plots show the primary noise policy; set to "shared_fresh" to inspect the ablation
# (only meaningful if it was included in NOISE_POLICIES when the classes were run).
PLOT_POLICY = NOISE_POLICY_PRIMARY


def get_metric(class_label, s, c, alpha, key, policy=None):
    """One metric, averaged over the class's M base trajectories."""
    entries = all_metrics[class_label]["exp3"]
    keys = [metric_key(s, c, alpha, m=m, policy=policy or PLOT_POLICY) for m in range(M)]
    vals = [entries[k][key] for k in keys if k in entries]
    if not vals:
        raise KeyError(f"{keys[0]} missing for class {class_label}")
    return float(np.mean(vals))


def tier_mean_std(cls_list, s, c, alpha, key):
    vals = np.array([get_metric(cl, s, c, alpha, key) for cl in cls_list if cl in all_metrics])
    if len(vals) == 0:
        return np.nan, np.nan
    return float(vals.mean()), float(vals.std(ddof=0))


# Sensitivity vs checkpoint by tier (avg over alphas; mean±std across classes)
for key in metric_keys:
    fig, axes = plt.subplots(1, len(pca_components), figsize=(4.5 * len(pca_components), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, c in zip(axes, pca_components):
        for tier_name, cls_list in tiers.items():
            means, stds = [], []
            for s in checkpoint_steps:
                per_class = []
                for cl in cls_list:
                    if cl not in all_metrics:
                        continue
                    per_class.append(np.mean([get_metric(cl, s, c, a, key) for a in alphas]))
                per_class = np.array(per_class)
                means.append(per_class.mean() if len(per_class) else np.nan)
                stds.append(per_class.std(ddof=0) if len(per_class) else np.nan)
            means, stds = np.array(means), np.array(stds)
            ax.plot(list(checkpoint_steps), means, marker="o", color=tier_colors[tier_name], label=tier_name)
            ax.fill_between(
                list(checkpoint_steps), means - stds, means + stds,
                color=tier_colors[tier_name], alpha=0.2,
            )
        ax.set_xlabel("checkpoint reverse-step s")
        ax.set_title(f"PC{c + 1}")
        ax.legend(fontsize=8)
    axes[0].set_ylabel(key)
    fig.suptitle(
        f"Exp3 PCA sensitivity by KID tier (mean +/- std over classes, avg over alpha) | {key}",
        y=1.03,
    )
    plt.tight_layout()
    fig_path = RESULTS_DIR / f"tier_sensitivity_vs_checkpoint_{key}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved {fig_path}")
    plt.show()


In [ ]:
# Heatmap of tier-mean sensitivity: rows=checkpoint, cols=(PC x alpha), one panel per tier
combo_labels = [f"PC{c + 1}\na={a:+.1f}" for c in pca_components for a in alphas]

for key in metric_keys:
    fig, axes = plt.subplots(1, 3, figsize=(5 * 3, 4))
    mats = []
    for tier_name, cls_list in tiers.items():
        mat = np.array(
            [
                [
                    tier_mean_std(cls_list, s, c, a, key)[0]
                    for c in pca_components
                    for a in alphas
                ]
                for s in checkpoint_steps
            ]
        )
        mats.append(mat)
    vmin = np.nanmin([m.min() for m in mats])
    vmax = np.nanmax([m.max() for m in mats])
    for ax, (tier_name, _), mat in zip(axes, tiers.items(), mats):
        im = ax.imshow(mat, aspect="auto", cmap="magma", vmin=vmin, vmax=vmax)
        ax.set_xticks(range(len(combo_labels)))
        ax.set_xticklabels(combo_labels, fontsize=7, rotation=45, ha="right")
        ax.set_yticks(range(len(checkpoint_steps)))
        ax.set_yticklabels([str(s) for s in checkpoint_steps])
        ax.set_ylabel("checkpoint s")
        ax.set_title(tier_name)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"Tier-mean {key} heatmaps", y=1.05)
    plt.tight_layout()
    fig_path = RESULTS_DIR / f"tier_heatmaps_{key}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved {fig_path}")
    plt.show()


In [ ]:
# Boxplot: per-class mean sensitivity (avg over s, PC, alpha) colored by tier
fig, axes = plt.subplots(1, len(metric_keys), figsize=(3.8 * len(metric_keys), 4.5))
axes = np.atleast_1d(axes)

for ax, key in zip(axes, metric_keys):
    data, labels, colors = [], [], []
    for tier_name, cls_list in tiers.items():
        vals = []
        for cl in cls_list:
            if cl not in all_metrics:
                continue
            vals.append(
                np.mean(
                    [
                        get_metric(cl, s, c, a, key)
                        for s in checkpoint_steps
                        for c in pca_components
                        for a in alphas
                    ]
                )
            )
        data.append(vals)
        labels.append(f"{tier_name}\n(n={len(vals)})")
        colors.append(tier_colors[tier_name])
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
    ax.set_ylabel(key)
    ax.set_title(key)
fig.suptitle("Per-class mean PCA sensitivity by KID difficulty tier", y=1.02)
plt.tight_layout()
fig_path = RESULTS_DIR / "tier_boxplots.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Saved {fig_path}")
plt.show()


# 8b. Cross-experiment axes

Two views that exist so these numbers can be *combined* with Exp 1 and Exp 2 rather than only read on their own:

- **Sensitivity vs normalised magnitude** — the x-axis is the shift's per-element RMS divided by the local latent std at that checkpoint, the common unit agreed for all perturbation types. Exp 1/2's absolute-unit perturbations (gaussian, blur, mask, geometric) convert onto exactly these axes, so all five perturbation types can be drawn as curves on one plot.
- **Sensitivity vs KID difficulty** — one Spearman $\rho$ per metric across all 45 classes, pooled over checkpoints, PCs and magnitudes, plus a per-checkpoint breakdown. Pooling is the shared version of the test each experiment was running separately, and it has far more power at no extra compute.

In [ ]:
# Sensitivity vs perturbation magnitude in local-latent-std units [shared].
# alpha is relative to each PC's own spread, so the same alpha is a different physical
# shift at different checkpoints and components; alpha_std_units removes that.
pc_styles = ("-", "--", ":", "-.")

for key in metric_keys:
    fig, axes = plt.subplots(
        1, len(checkpoint_steps), figsize=(3.4 * len(checkpoint_steps), 3.8), sharey=True
    )
    axes = np.atleast_1d(axes)
    for ax_i, (ax, s) in enumerate(zip(axes, checkpoint_steps)):
        for tier_name, cls_list in tiers.items():
            for j, c in enumerate(pca_components):
                xs, ys = [], []
                for a in alphas:
                    present = [cl for cl in cls_list if cl in all_metrics]
                    if not present:
                        continue
                    xs.append(np.mean([get_metric(cl, s, c, a, "alpha_std_units") for cl in present]))
                    ys.append(np.mean([get_metric(cl, s, c, a, key) for cl in present]))
                if not xs:
                    continue
                order = np.argsort(xs)
                ax.plot(
                    np.array(xs)[order],
                    np.array(ys)[order],
                    marker="o",
                    ms=3.5,
                    lw=1.2,
                    ls=pc_styles[j % len(pc_styles)],
                    color=tier_colors[tier_name],
                    label=f"{tier_name} PC{c + 1}" if ax_i == 0 else None,
                )
        ax.set_xscale("log")
        ax.set_xlabel("shift RMS / latent std")
        ax.set_title(f"s={s}")
    axes[0].set_ylabel(key)
    axes[0].legend(fontsize=6, ncol=3)
    fig.suptitle(
        f"Exp3 sensitivity vs normalised perturbation magnitude ({PLOT_POLICY}) | {key}", y=1.03
    )
    plt.tight_layout()
    fig_path = RESULTS_DIR / f"tier_sensitivity_vs_normalized_magnitude_{key}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved {fig_path}")
    plt.show()

In [ ]:
# Does PCA-directional sensitivity track per-class KID difficulty?
from scipy.stats import spearmanr

classes_present = [c for c in class_labels if c in all_metrics]
kid_vals = np.array([kid_per_class[c]["kid"] for c in classes_present])


def class_sensitivity(class_label, key, steps):
    return float(
        np.mean(
            [
                get_metric(class_label, s, c, a, key)
                for s in steps
                for c in pca_components
                for a in alphas
            ]
        )
    )


pooled, per_checkpoint = [], {}
for key in metric_keys:
    sens = np.array([class_sensitivity(cl, key, checkpoint_steps) for cl in classes_present])
    rho, p_value = spearmanr(kid_vals, sens)
    pooled.append({"metric": key, "rho": float(rho), "p_value": float(p_value)})
    print(f"{key:<16} rho={rho:+.3f}  p={p_value:.3g}  (n={len(classes_present)} classes)")

    per_checkpoint[key] = {}
    for s in checkpoint_steps:
        sens_s = np.array([class_sensitivity(cl, key, (s,)) for cl in classes_present])
        rho_s, p_s = spearmanr(kid_vals, sens_s)
        per_checkpoint[key][str(s)] = {"rho": float(rho_s), "p_value": float(p_s)}

corr_path = RESULTS_DIR / "kid_sensitivity_correlation.json"
with open(corr_path, "w") as f:
    json.dump(
        {
            "noise_policy": PLOT_POLICY,
            "n_classes": len(classes_present),
            "classes": classes_present,
            "pooled": pooled,
            "per_checkpoint": per_checkpoint,
        },
        f,
        indent=2,
    )
print(f"\nSaved {corr_path}")

# 9. Spotlight class visuals

Metrics for every class are already in `metrics_pca_perturbation.json` (all checkpoints). This section only **renders PNGs** at `spotlight_show_steps` (default s=125, 150 on the shared grid) for the three spotlight classes, using base trajectory `m=0` and the primary noise policy — a short denoise pass, **not** a full experiment re-run.


In [ ]:
from IPython.display import display

# Render PNGs at s=100/150 if missing (uses cached traj + PCA ensemble only)
for tier_name, cl in spotlight_classes.items():
    save_spotlight_visuals(cl)

for tier_name, cl in spotlight_classes.items():
    samples_dir = RESULTS_DIR / f"class_{cl:04d}"
    print(
        f"\n=== Spotlight: {tier_name} | class {cl} ({class_names.get(cl, '?')}) "
        f"| KID={kid_per_class[cl]['kid']:.4f} | s={list(spotlight_show_steps)} ==="
    )
    paths = []
    for s in spotlight_show_steps:
        paths.extend(sorted(samples_dir.glob(f"grid_s{s}_pc*.png")))
        paths.extend(sorted(samples_dir.glob(f"diffmap_s{s}_pc*.png")))
    for path in paths:
        print(path)
        display(Image.open(path))
